In [ ]:
# Colab setup - run this cell first
import sys, os
from pathlib import Path

if 'google.colab' in sys.modules:
    repo_dir = '/content/CompMathAndAICourse/'
    if not os.path.exists(repo_dir):
        !git clone https://github.com/lruthotto/CompMathAndAICourse.git {repo_dir}
        %pip install -q jax jaxlib
    
    NOTEBOOK_DIR = Path(repo_dir) / 'code' / '09-inverse-problems'
    sys.path.insert(0, str(NOTEBOOK_DIR))
    print(f"Running on Colab, repo cloned to {repo_dir}")
else:
    # Local execution
    try:
        import IPython
        notebook_path = IPython.extract_module_locals()[1]['__vsc_ipynb_file__']
        NOTEBOOK_DIR = Path(notebook_path).parent
    except (KeyError, AttributeError, TypeError):
        NOTEBOOK_DIR = Path.cwd()
    
    if str(NOTEBOOK_DIR) not in sys.path:
        sys.path.insert(0, str(NOTEBOOK_DIR))
    print(f"Running locally from {NOTEBOOK_DIR}")

# Diffusion Posterior Sampling (DPS) for Inverse Problems

**Course:** Computational Mathematics and AI  
**Lecture 9:** Machine Learning for Inverse Problems

## Overview

This notebook demonstrates **Diffusion Posterior Sampling (DPS)** on a simple 1D Bayesian inference problem.

### Problem Setup

We have:
- **Prior** $\pi(x)$: A 3-component Gaussian Mixture Model (GMM) with modes at $\{-2.5, 0.0, 1.5\}$
- **Likelihood** $p(y|x)$: Gaussian centered at observed value $y_{obs} = 0.5$ with noise $\sigma_y = 0.8$

### Goal

Sample from the **posterior** distribution:
$$\pi(x|y) \propto p(y|x) \cdot \pi(x)$$

For GMM prior and Gaussian likelihood, the posterior is analytically a GMM with updated parameters.

### DPS Algorithm

DPS uses a pre-trained diffusion model (score network) to sample from posteriors:

$$\nabla_x \log \pi(x|y) = \underbrace{\nabla_x \log \pi(x)}_{\text{prior score}} + \underbrace{\nabla_x \log p(y|x)}_{\text{likelihood gradient}}$$

The key insight: we can use an **analytically known score** for the GMM prior and combine it with the likelihood gradient.

## Learning Objectives

1. Understand the DPS algorithm and posterior score decomposition
2. Compute analytical GMM posteriors for comparison
3. Compare DPS samples with true posterior samples
4. Visualize the space-time evolution of diffusion sampling

In [ ]:
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend

import jax
import jax.numpy as jnp
from jax import random, grad, vmap
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Import GMM utilities from local utils
from utils import GaussianMixtureDistribution, NoiseSchedule

# Set up JAX
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")
print(f"JAX devices: {jax.devices()}")
print(f"JAX version: {jax.__version__}")

# Create output directory for figures
OUTPUT_DIR = NOTEBOOK_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Output directory: {OUTPUT_DIR.absolute()}")

# Master PRNG key for reproducibility
MASTER_KEY = random.PRNGKey(42)

## Configuration

Define the prior GMM, likelihood, and diffusion parameters.

In [ ]:
# Prior: 3-component Gaussian Mixture Model
prior_config = {
    'means': jnp.array([-2.5, 0.0, 1.5]),
    'stds': jnp.array([0.3, 0.4, 0.3]),
    'weights': jnp.array([0.3, 0.5, 0.2]),
}

# Likelihood: Gaussian observation model
# y = x + noise, where noise ~ N(0, sigma_y^2)
likelihood_config = {
    'y_obs': 0.5,       # Observed value
    'sigma_y': 0.8,     # Observation noise (fairly wide to allow multiple modes)
}

# VP Diffusion schedule
# Note: NoiseSchedule expects t in [0, 1], so T=1.0
# With beta_min=0.1, beta_max=20.0, at T=1: alpha~0.007, sigma~0.99998
diffusion_config = {
    'beta_min': 0.1,
    'beta_max': 20.0,
    'T': 1.0,             # T=1 (schedule expects t in [0,1])
    'n_steps': 1000,      # Number of discretization steps
}

# Sampling
sampling_config = {
    'n_samples': 5000,      # Number of posterior samples
    'guidance_scale': 1.0,  # Likelihood guidance strength (lambda)
}

print("Configuration:")
print(f"  Prior means: {prior_config['means']}")
print(f"  Prior stds: {prior_config['stds']}")
print(f"  Prior weights: {prior_config['weights']}")
print(f"  y_obs: {likelihood_config['y_obs']}")
print(f"  sigma_y: {likelihood_config['sigma_y']}")
print(f"  Diffusion T: {diffusion_config['T']}")
print(f"  Number of samples: {sampling_config['n_samples']}")

## 1. Define the Prior Distribution

The prior is a 3-component Gaussian Mixture Model:
$$\pi(x) = \sum_{k=1}^{3} w_k \mathcal{N}(x; \mu_k, \sigma_k^2)$$

In [4]:
# Create prior distribution
prior = GaussianMixtureDistribution(
    means=prior_config['means'],
    stds=prior_config['stds'],
    weights=prior_config['weights']
)

# Visualize prior
x_grid = jnp.linspace(-5, 5, 500)
prior_pdf = prior.pdf(x_grid)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_grid, prior_pdf, 'b-', linewidth=2, label='Prior $\\pi(x)$')
ax.fill_between(x_grid, prior_pdf, alpha=0.3)
ax.axvline(likelihood_config['y_obs'], color='r', linestyle='--', linewidth=2, 
           label=f"$y_{{obs}} = {likelihood_config['y_obs']}$")
ax.set_xlabel('$x$', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Prior Distribution (3-component GMM)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-5, 5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'prior_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {OUTPUT_DIR / 'prior_distribution.png'}")

Saved: figures/prior_distribution.png


## 2. Compute Analytical Posterior

For a GMM prior and Gaussian likelihood, the posterior is also a GMM.

### Bayes' Rule for GMM

Given:
- Prior component $k$: $\mathcal{N}(\mu_k, \sigma_k^2)$ with weight $w_k$
- Likelihood: $p(y|x) = \mathcal{N}(y; x, \sigma_y^2)$

The posterior component $k$ is:
$$\pi_k(x|y) = \mathcal{N}(x; \tilde{\mu}_k, \tilde{\sigma}_k^2)$$

where:
$$\tilde{\sigma}_k^2 = \frac{1}{1/\sigma_k^2 + 1/\sigma_y^2} = \frac{\sigma_k^2 \sigma_y^2}{\sigma_k^2 + \sigma_y^2}$$
$$\tilde{\mu}_k = \tilde{\sigma}_k^2 \left( \frac{\mu_k}{\sigma_k^2} + \frac{y}{\sigma_y^2} \right)$$

The updated weights are:
$$\tilde{w}_k \propto w_k \cdot \mathcal{N}(y; \mu_k, \sigma_k^2 + \sigma_y^2)$$

In [5]:
def compute_gmm_posterior(prior_means, prior_stds, prior_weights, y_obs, sigma_y):
    """
    Compute the analytical posterior GMM parameters.
    
    For each component k:
    - Posterior variance: 1/(1/sigma_k^2 + 1/sigma_y^2)
    - Posterior mean: posterior_var * (mu_k/sigma_k^2 + y_obs/sigma_y^2)
    - Posterior weight: proportional to w_k * N(y_obs; mu_k, sigma_k^2 + sigma_y^2)
    """
    n_components = len(prior_means)
    
    # Posterior variances (precision addition)
    prior_var = prior_stds ** 2
    sigma_y_sq = sigma_y ** 2
    posterior_var = 1.0 / (1.0 / prior_var + 1.0 / sigma_y_sq)
    posterior_stds = jnp.sqrt(posterior_var)
    
    # Posterior means (precision-weighted average)
    posterior_means = posterior_var * (prior_means / prior_var + y_obs / sigma_y_sq)
    
    # Posterior weights (evidence for each component)
    # p(y | component k) = N(y; mu_k, sigma_k^2 + sigma_y^2)
    evidence_var = prior_var + sigma_y_sq
    log_evidence = -0.5 * jnp.log(2 * jnp.pi * evidence_var) - 0.5 * (y_obs - prior_means)**2 / evidence_var
    log_unnorm_weights = jnp.log(prior_weights) + log_evidence
    log_norm = jax.scipy.special.logsumexp(log_unnorm_weights)
    posterior_weights = jnp.exp(log_unnorm_weights - log_norm)
    
    return posterior_means, posterior_stds, posterior_weights

# Compute posterior parameters
post_means, post_stds, post_weights = compute_gmm_posterior(
    prior_config['means'],
    prior_config['stds'],
    prior_config['weights'],
    likelihood_config['y_obs'],
    likelihood_config['sigma_y']
)

print("\nAnalytical Posterior GMM Parameters:")
print(f"  Posterior means: {post_means}")
print(f"  Posterior stds: {post_stds}")
print(f"  Posterior weights: {post_weights}")
print(f"  (weights sum: {jnp.sum(post_weights):.6f})")

# Create posterior distribution
posterior = GaussianMixtureDistribution(
    means=post_means,
    stds=post_stds,
    weights=post_weights
)


Analytical Posterior GMM Parameters:
  Posterior means: [-2.13013699  0.1         1.37671233]
  Posterior stds: [0.28089875 0.35777088 0.28089875]
  Posterior weights: [0.00123709 0.80106289 0.19770002]
  (weights sum: 1.000000)


In [6]:
# Visualize prior vs posterior
posterior_pdf = posterior.pdf(x_grid)

# Also compute likelihood (unnormalized, for visualization)
likelihood_unnorm = jnp.exp(-0.5 * (x_grid - likelihood_config['y_obs'])**2 / likelihood_config['sigma_y']**2)
likelihood_scaled = likelihood_unnorm * jnp.max(prior_pdf) / jnp.max(likelihood_unnorm) * 0.5

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_grid, prior_pdf, 'b-', linewidth=2, label='Prior $\\pi(x)$')
ax.plot(x_grid, likelihood_scaled, 'g--', linewidth=2, label='Likelihood $p(y|x)$ (scaled)')
ax.plot(x_grid, posterior_pdf, 'r-', linewidth=2.5, label='Posterior $\\pi(x|y)$')
ax.axvline(likelihood_config['y_obs'], color='gray', linestyle=':', linewidth=1.5, 
           label=f"$y_{{obs}} = {likelihood_config['y_obs']}$")

ax.set_xlabel('$x$', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Bayesian Update: Prior × Likelihood ∝ Posterior', fontsize=14)
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(-5, 5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'prior_vs_posterior.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {OUTPUT_DIR / 'prior_vs_posterior.png'}")

Saved: figures/prior_vs_posterior.png


## 3. VP Diffusion Forward Process

The VP-SDE forward process adds noise according to:
$$dx = -\frac{1}{2}\beta(t) x \, dt + \sqrt{\beta(t)} \, dW$$

For a Gaussian mixture prior, the marginal at time $t$ is also a mixture:
$$p_t(x) = \sum_k w_k \mathcal{N}(x; \alpha(t)\mu_k, \alpha(t)^2\sigma_k^2 + \sigma(t)^2)$$

where $\alpha(t), \sigma(t)$ come from the noise schedule.

In [7]:
# Create noise schedule
schedule = NoiseSchedule(
    beta_min=diffusion_config['beta_min'],
    beta_max=diffusion_config['beta_max'],
    schedule_type='linear'
)

# Visualize noise schedule coefficients
T = diffusion_config['T']
t_vis = jnp.linspace(0, T, 100)

alpha_vis = []
sigma_vis = []
for t in t_vis:
    a, s = schedule.get_coefficients(jnp.array(t))
    alpha_vis.append(float(a))
    sigma_vis.append(float(s))

alpha_vis = jnp.array(alpha_vis)
sigma_vis = jnp.array(sigma_vis)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t_vis, alpha_vis, 'b-', linewidth=2, label=r'$\alpha(t)$ (signal)')
axes[0].plot(t_vis, sigma_vis, 'r-', linewidth=2, label=r'$\sigma(t)$ (noise)')
axes[0].set_xlabel('$t$', fontsize=12)
axes[0].set_ylabel('Coefficient', fontsize=12)
axes[0].set_title('VP Noise Schedule Coefficients', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Show SNR
snr = alpha_vis**2 / (sigma_vis**2 + 1e-10)
axes[1].semilogy(t_vis, snr + 1e-10, 'g-', linewidth=2)
axes[1].set_xlabel('$t$', fontsize=12)
axes[1].set_ylabel('SNR = $\\alpha^2/\\sigma^2$', fontsize=12)
axes[1].set_title('Signal-to-Noise Ratio', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'noise_schedule.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {OUTPUT_DIR / 'noise_schedule.png'}")

# Check marginal at T
alpha_T, sigma_T = schedule.get_coefficients(jnp.array(T))
print(f"\nAt T={T}:")
print(f"  alpha(T) = {float(alpha_T):.6f}")
print(f"  sigma(T) = {float(sigma_T):.6f}")
print(f"  SNR = {float(alpha_T**2 / (sigma_T**2 + 1e-10)):.6f}")
print(f"  -> Prior at T is approximately N(0, 1)")

Saved: figures/noise_schedule.png

At T=1.0:
  alpha(T) = 0.006572
  sigma(T) = 0.999978
  SNR = 0.000043
  -> Prior at T is approximately N(0, 1)


## 4. Marginal Distribution Evolution

For the GMM prior, we can compute the marginal $p_t(x)$ at any time $t$ analytically.

In [8]:
def get_marginal_distribution(t, prior_means, prior_stds, prior_weights, schedule):
    """
    Compute the marginal GMM at time t.
    
    p_t(x) = sum_k w_k * N(x; alpha(t)*mu_k, alpha(t)^2*sigma_k^2 + sigma(t)^2)
    """
    alpha_t, sigma_t = schedule.get_coefficients(jnp.array(t))
    alpha_t = float(alpha_t)
    sigma_t = float(sigma_t)
    
    marginal_means = alpha_t * prior_means
    marginal_vars = alpha_t**2 * prior_stds**2 + sigma_t**2
    marginal_stds = jnp.sqrt(marginal_vars)
    
    return GaussianMixtureDistribution(marginal_means, marginal_stds, prior_weights)

def get_marginal_posterior_distribution(t, post_means, post_stds, post_weights, schedule):
    """
    Compute the marginal posterior GMM at time t.
    Same formula but starting from the posterior distribution.
    """
    alpha_t, sigma_t = schedule.get_coefficients(jnp.array(t))
    alpha_t = float(alpha_t)
    sigma_t = float(sigma_t)
    
    marginal_means = alpha_t * post_means
    marginal_vars = alpha_t**2 * post_stds**2 + sigma_t**2
    marginal_stds = jnp.sqrt(marginal_vars)
    
    return GaussianMixtureDistribution(marginal_means, marginal_stds, post_weights)

In [9]:
# Visualize marginal evolution for prior
times_to_show = [0, 0.1, 0.2, 0.4, 0.6, 1.0]
colors = plt.cm.viridis(jnp.linspace(0, 1, len(times_to_show)))

fig, ax = plt.subplots(figsize=(10, 5))

for i, t in enumerate(times_to_show):
    marginal = get_marginal_distribution(
        t, prior_config['means'], prior_config['stds'], prior_config['weights'], schedule
    )
    pdf = marginal.pdf(x_grid)
    ax.plot(x_grid, pdf, '-', color=colors[i], linewidth=2, label=f'$t = {t}$')

# Also show standard normal for reference
std_normal = jnp.exp(-0.5 * x_grid**2) / jnp.sqrt(2 * jnp.pi)
ax.plot(x_grid, std_normal, 'k--', linewidth=1.5, alpha=0.5, label='$\\mathcal{N}(0,1)$')

ax.set_xlabel('$x$', fontsize=12)
ax.set_ylabel('$p_t(x)$', fontsize=12)
ax.set_title('Forward Diffusion: Marginal Distribution Evolution', fontsize=14)
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(-5, 5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'marginal_evolution.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {OUTPUT_DIR / 'marginal_evolution.png'}")

Saved: figures/marginal_evolution.png


## 5. Score Functions

The **score** is the gradient of log-density: $s(x) = \nabla_x \log p(x)$

For DPS, we need:
1. **Prior score** at time $t$: $s_t^{\text{prior}}(x) = \nabla_x \log p_t(x)$
2. **Likelihood gradient**: $\nabla_x \log p(y|x)$

The posterior score is: $s_t^{\text{post}}(x) = s_t^{\text{prior}}(x) + \lambda \nabla_x \log p(y|\hat{x}_0)$

where $\hat{x}_0$ is the Tweedie estimate of the clean sample.

In [10]:
def prior_score_t(x, t, prior_means, prior_stds, prior_weights, schedule):
    """
    Compute the score of the marginal prior at time t.
    This is what a trained score network would approximate.
    """
    marginal = get_marginal_distribution(t, prior_means, prior_stds, prior_weights, schedule)
    return marginal.score(x)

def likelihood_gradient(x, y_obs, sigma_y):
    """
    Gradient of log p(y|x) for Gaussian observation model.
    p(y|x) = N(y; x, sigma_y^2)
    log p(y|x) = -0.5 * (y - x)^2 / sigma_y^2 + const
    d/dx log p(y|x) = (y - x) / sigma_y^2
    """
    return (y_obs - x) / sigma_y**2

def tweedie_estimate(x_t, t, prior_means, prior_stds, prior_weights, schedule):
    """
    Tweedie's formula to estimate x_0 from x_t:
    E[x_0 | x_t] = (x_t + sigma_t^2 * score(x_t, t)) / alpha_t
    """
    alpha_t, sigma_t = schedule.get_coefficients(jnp.array(t))
    alpha_t = float(alpha_t)
    sigma_t = float(sigma_t)
    
    score_t = prior_score_t(x_t, t, prior_means, prior_stds, prior_weights, schedule)
    x0_hat = (x_t + sigma_t**2 * score_t) / alpha_t
    return x0_hat

In [11]:
# Visualize score functions in space-time domain
# Compute prior and posterior scores across (t, x) space

# Use log-spaced time for better visualization of early dynamics
n_t_score = 50
n_x_score = 100
t_score = jnp.logspace(-2, 0, n_t_score)  # log-spaced from 0.01 to 1.0
x_score = jnp.linspace(-4, 4, n_x_score)

# Compute prior score field
prior_score_field = jnp.zeros((n_t_score, n_x_score))
for i, t in enumerate(t_score):
    marginal = get_marginal_distribution(
        float(t), prior_config['means'], prior_config['stds'], prior_config['weights'], schedule
    )
    prior_score_field = prior_score_field.at[i].set(marginal.score(x_score))

# Compute posterior score field
posterior_score_field = jnp.zeros((n_t_score, n_x_score))
for i, t in enumerate(t_score):
    marginal_post = get_marginal_posterior_distribution(
        float(t), post_means, post_stds, post_weights, schedule
    )
    posterior_score_field = posterior_score_field.at[i].set(marginal_post.score(x_score))

# Plot prior score in space-time (using viridis colormap)
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.pcolormesh(
    np.array(t_score), np.array(x_score), np.array(prior_score_field.T),
    cmap='viridis', shading='auto'
)
ax.set_xscale('log')
ax.set_xlabel('$t$ (log scale)', fontsize=12)
ax.set_ylabel('$x$', fontsize=12)
ax.set_title('Prior Score Field $\\nabla_x \\log p_t(x)$', fontsize=14)
plt.colorbar(im, ax=ax, label='Score')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'prior_score_spacetime.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {OUTPUT_DIR / 'prior_score_spacetime.png'}")

# Plot posterior score in space-time (using viridis colormap)
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.pcolormesh(
    np.array(t_score), np.array(x_score), np.array(posterior_score_field.T),
    cmap='viridis', shading='auto'
)
ax.set_xscale('log')
ax.set_xlabel('$t$ (log scale)', fontsize=12)
ax.set_ylabel('$x$', fontsize=12)
ax.set_title('Posterior Score Field $\\nabla_x \\log p_t(x|y)$', fontsize=14)
plt.colorbar(im, ax=ax, label='Score')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'posterior_score_spacetime.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {OUTPUT_DIR / 'posterior_score_spacetime.png'}")

Saved: figures/prior_score_spacetime.png


Saved: figures/posterior_score_spacetime.png


## 6. DPS Sampling Algorithm

DPS modifies the reverse diffusion by adding likelihood guidance:

**Standard reverse SDE:**
$$dx = \left[ -\frac{1}{2}\beta(t) x - \beta(t) s_t^{\text{prior}}(x) \right] dt + \sqrt{\beta(t)} d\bar{W}$$

**DPS reverse SDE:**
$$dx = \left[ -\frac{1}{2}\beta(t) x - \beta(t) s_t^{\text{prior}}(x) + \lambda \nabla_x \log p(y|\hat{x}_0) \right] dt + \sqrt{\beta(t)} d\bar{W}$$

where $\hat{x}_0$ is the Tweedie estimate.

In [12]:
def dps_sample(key, n_samples, prior_means, prior_stds, prior_weights, 
               y_obs, sigma_y, schedule, T, n_steps, guidance_scale=1.0,
               return_trajectory=False):
    """
    DPS sampling using Euler-Maruyama discretization of the reverse SDE.
    
    VP Forward SDE: dx = -0.5 β(t) x dt + √β(t) dW
    VP Reverse SDE: dx = [-0.5 β(t) x - β(t) ∇log p_t(x)] dt + √β(t) d̄W (backward in time)
    
    Since we step forward (dt > 0) but go backward in time (t: T→0), the effective drift is:
    drift = 0.5 β x + β ∇log p_t(x)
    
    DPS adds likelihood guidance via Tweedie's estimate.
    
    IMPORTANT: At large t (small alpha), the Tweedie estimate becomes unreliable and
    the 1/alpha scaling of the likelihood gradient can cause instability. We only
    apply likelihood guidance when alpha > threshold (SNR is reasonable).
    """
    # Time discretization (backward from T to 0)
    dt = T / n_steps  # positive step size
    times = jnp.linspace(T, 0, n_steps + 1)
    
    # Initialize from approximate prior at T (close to N(0,1))
    key, subkey = random.split(key)
    x = random.normal(subkey, (n_samples,))
    
    trajectory = [x] if return_trajectory else None
    
    # Threshold for applying likelihood guidance
    # Only apply when alpha > 0.1 (SNR > ~0.01), i.e., when signal is still present
    alpha_threshold = 0.1
    
    # Reverse-time iteration (t goes from T to 0)
    for i in range(n_steps):
        t = float(times[i])
        
        # Get noise schedule coefficients
        alpha_t, sigma_t = schedule.get_coefficients(jnp.array(t))
        alpha_t = float(alpha_t)
        sigma_t = float(sigma_t)
        beta_t = float(schedule.beta(jnp.array(t)))
        
        # Prior score at current x and t
        s_prior = prior_score_t(x, t, prior_means, prior_stds, prior_weights, schedule)
        
        # Only apply likelihood guidance when alpha is large enough
        # At small alpha (large t), the Tweedie estimate is unreliable
        if alpha_t > alpha_threshold:
            # Tweedie estimate of x_0
            x0_hat = (x + sigma_t**2 * s_prior) / alpha_t
            x0_hat = jnp.clip(x0_hat, -10, 10)
            
            # Likelihood gradient at x0_hat
            lik_grad = likelihood_gradient(x0_hat, y_obs, sigma_y)
            
            # Scale by 1/alpha_t (chain rule for d(x0_hat)/d(x_t))
            lik_grad_scaled = lik_grad / alpha_t * guidance_scale
        else:
            # No likelihood guidance at large t (pure prior diffusion)
            lik_grad_scaled = 0.0
        
        # Reverse SDE drift
        drift = 0.5 * beta_t * x + beta_t * s_prior + lik_grad_scaled
        diffusion = jnp.sqrt(beta_t)
        
        # Euler-Maruyama step
        key, subkey = random.split(key)
        noise = random.normal(subkey, x.shape)
        x = x + drift * dt + diffusion * jnp.sqrt(dt) * noise
        
        if return_trajectory:
            trajectory.append(x)
    
    if return_trajectory:
        return x, jnp.stack(trajectory, axis=0)
    return x

print("DPS sampler defined")

DPS sampler defined


In [13]:
def standard_sde_sample(key, n_samples, post_means, post_stds, post_weights, 
                        schedule, T, n_steps, return_trajectory=False):
    """
    Standard reverse SDE sampling using the TRUE POSTERIOR score.
    
    This serves as ground truth - what sampling should produce if we
    had access to the posterior score directly (not via DPS approximation).
    
    VP Reverse SDE drift (stepping forward while time decreases):
    drift = 0.5 β x + β ∇log p_t(x)
    """
    dt = T / n_steps
    times = jnp.linspace(T, 0, n_steps + 1)
    
    # Initialize from N(0,1)
    key, subkey = random.split(key)
    x = random.normal(subkey, (n_samples,))
    
    trajectory = [x] if return_trajectory else None
    
    for i in range(n_steps):
        t = float(times[i])
            
        beta_t = float(schedule.beta(jnp.array(t)))
        
        # Use the TRUE POSTERIOR score at time t
        marginal_post = get_marginal_posterior_distribution(
            t, post_means, post_stds, post_weights, schedule
        )
        s_post = marginal_post.score(x)
        
        # Reverse SDE drift (positive sign for score since we step forward in dt)
        drift = 0.5 * beta_t * x + beta_t * s_post
        diffusion = jnp.sqrt(beta_t)
        
        key, subkey = random.split(key)
        noise = random.normal(subkey, x.shape)
        x = x + drift * dt + diffusion * jnp.sqrt(dt) * noise
        
        if return_trajectory:
            trajectory.append(x)
    
    if return_trajectory:
        return x, jnp.stack(trajectory, axis=0)
    return x

print("Standard SDE sampler defined")

Standard SDE sampler defined


## 7. Generate Samples and Compare

We'll compare:
1. **DPS samples**: Using prior score + likelihood guidance
2. **True posterior samples**: Using analytical posterior score (ground truth)
3. **Direct samples**: From analytical posterior GMM

In [14]:
n_samples = sampling_config['n_samples']
T = diffusion_config['T']
n_steps = diffusion_config['n_steps']

print(f"Generating {n_samples} samples...")

# Generate DPS samples with trajectory
key, subkey = random.split(MASTER_KEY)
print("  DPS sampling...")
dps_samples, dps_trajectory = dps_sample(
    subkey, n_samples,
    prior_config['means'], prior_config['stds'], prior_config['weights'],
    likelihood_config['y_obs'], likelihood_config['sigma_y'],
    schedule, T, n_steps,
    guidance_scale=sampling_config['guidance_scale'],
    return_trajectory=True
)
print(f"    DPS samples range: [{float(dps_samples.min()):.3f}, {float(dps_samples.max()):.3f}]")

# Generate ground truth samples using true posterior score
key, subkey = random.split(key)
print("  Standard SDE sampling (true posterior)...")
sde_samples, sde_trajectory = standard_sde_sample(
    subkey, n_samples,
    post_means, post_stds, post_weights,
    schedule, T, n_steps,
    return_trajectory=True
)
print(f"    SDE samples range: [{float(sde_samples.min()):.3f}, {float(sde_samples.max()):.3f}]")

# Direct samples from analytical posterior
key, subkey = random.split(key)
print("  Direct posterior sampling...")
direct_samples = posterior.sample(subkey, n_samples)
print(f"    Direct samples range: [{float(direct_samples.min()):.3f}, {float(direct_samples.max()):.3f}]")

print("\nDone!")

Generating 5000 samples...
  DPS sampling...


    DPS samples range: [-2.649, 2.214]
  Standard SDE sampling (true posterior)...


    SDE samples range: [-2.902, 2.262]
  Direct posterior sampling...


    Direct samples range: [-2.746, 2.253]

Done!


## 8. Compare Histograms

In [15]:
# Create histogram plots - separate figures for DPS and SDE
# Blue for DPS, Red for SDE, Black for true posterior

bins = jnp.linspace(-4, 4, 60)

# DPS samples histogram
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(np.array(dps_samples), bins=np.array(bins), density=True, alpha=0.7, 
        color='blue', label='DPS samples', edgecolor='darkblue')
ax.plot(x_grid, posterior_pdf, 'k-', linewidth=2.5, label='True posterior')
ax.set_xlabel('$x$', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('DPS Samples vs True Posterior', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-4, 4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'histogram_dps.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {OUTPUT_DIR / 'histogram_dps.png'}")

# SDE samples histogram (true posterior score)
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(np.array(sde_samples), bins=np.array(bins), density=True, alpha=0.7, 
        color='red', label='SDE samples', edgecolor='darkred')
ax.plot(x_grid, posterior_pdf, 'k-', linewidth=2.5, label='True posterior')
ax.set_xlabel('$x$', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Standard SDE (True Score) vs True Posterior', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-4, 4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'histogram_sde.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {OUTPUT_DIR / 'histogram_sde.png'}")

Saved: figures/histogram_dps.png


Saved: figures/histogram_sde.png


In [16]:
# Compute sample statistics
def compute_sample_stats(samples, name):
    """Compute mean, std, and mode estimates."""
    mean = float(jnp.mean(samples))
    std = float(jnp.std(samples))
    
    # KDE mode estimate
    from scipy.stats import gaussian_kde
    kde = gaussian_kde(np.array(samples))
    x_fine = np.linspace(-4, 4, 1000)
    mode = x_fine[np.argmax(kde(x_fine))]
    
    print(f"{name}:")
    print(f"  Mean: {mean:.4f}")
    print(f"  Std:  {std:.4f}")
    print(f"  Mode: {mode:.4f}")
    return mean, std, mode

print("\nSample Statistics:")
print("=" * 40)
dps_stats = compute_sample_stats(dps_samples, "DPS")
print()
sde_stats = compute_sample_stats(sde_samples, "SDE (true score)")
print()
direct_stats = compute_sample_stats(direct_samples, "Direct GMM")

# True posterior stats
print()
print("True Posterior:")
true_mean = float(jnp.sum(post_weights * post_means))
true_var = float(jnp.sum(post_weights * (post_stds**2 + post_means**2)) - true_mean**2)
print(f"  Mean: {true_mean:.4f}")
print(f"  Std:  {jnp.sqrt(true_var):.4f}")


Sample Statistics:


DPS:
  Mean: 0.3677
  Std:  0.8207
  Mode: 0.1161

SDE (true score):
  Mean: 0.3606
  Std:  0.6233
  Mode: 0.0761

Direct GMM:
  Mean: 0.3514
  Std:  0.6195
  Mode: 0.0921

True Posterior:
  Mean: 0.3496
  Std:  0.6199


## 9. Space-Time Density Visualization

Visualize the evolution of sample density over the reverse diffusion process.

In [17]:
def compute_spacetime_density_log(trajectory, x_bins, n_t_bins=100):
    """
    Compute 2D histogram of trajectory in (log t, x) space.
    
    Args:
        trajectory: (n_times, n_samples) array
        x_bins: Bin edges for x
        n_t_bins: Number of time bins
    
    Returns:
        density: 2D density array
        t_log_bins: log10(t) bin edges
    """
    n_times, n_samples = trajectory.shape
    
    # Time points corresponding to trajectory (backward: T to ~0)
    # Avoid exact 0 by using small epsilon
    times = jnp.linspace(T, T/n_times, n_times)
    
    # Convert to log scale
    times_log = jnp.log10(times)
    
    # Create log-spaced time bins
    t_log_min = float(jnp.min(times_log))
    t_log_max = float(jnp.max(times_log))
    t_log_bins = jnp.linspace(t_log_min, t_log_max, n_t_bins + 1)
    
    # Flatten trajectory for 2D histogram
    t_flat = jnp.repeat(times_log[:, None], n_samples, axis=1).flatten()
    x_flat = trajectory.flatten()
    
    # Compute 2D histogram in log-t space
    density, _, _ = jnp.histogram2d(
        np.array(t_flat), np.array(x_flat), 
        bins=[np.array(t_log_bins), np.array(x_bins)]
    )
    
    # Normalize each time slice
    density = density / (jnp.sum(density, axis=1, keepdims=True) + 1e-10)
    
    return density, t_log_bins

# Define x bins
x_bins = jnp.linspace(-5, 5, 100)

print("Computing space-time densities with log-scaled time...")
dps_density_log, t_log_bins = compute_spacetime_density_log(dps_trajectory, x_bins)
sde_density_log, _ = compute_spacetime_density_log(sde_trajectory, x_bins)
print(f"Trajectory shape: {dps_trajectory.shape}")
print(f"Time bins: {len(t_log_bins)-1} bins from log10(t)={float(t_log_bins[0]):.2f} to {float(t_log_bins[-1]):.2f}")
print("Done!")

Computing space-time densities with log-scaled time...


Trajectory shape: (1001, 5000)
Time bins: 100 bins from log10(t)=-3.00 to 0.00
Done!


In [18]:
# Create space-time density plots with log-scaled time and trajectory overlays
# Time axis in log scale, with white trajectory lines superimposed

n_traj_show = 30  # Number of trajectories to show

# Get trajectory times in log scale
n_times = dps_trajectory.shape[0]
times_traj = jnp.linspace(T, T/n_times, n_times)
times_traj_log = jnp.log10(times_traj)

# DPS space-time density with trajectories
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(
    np.array(dps_density_log.T),
    aspect='auto',
    origin='lower',
    extent=[float(t_log_bins[0]), float(t_log_bins[-1]), float(x_bins[0]), float(x_bins[-1])],
    cmap='viridis'
)
# Overlay white trajectory lines
for i in range(n_traj_show):
    ax.plot(times_traj_log, dps_trajectory[:, i], 'w-', alpha=0.4, linewidth=0.5)
ax.set_xlabel('$\\log_{10}(t)$', fontsize=12)
ax.set_ylabel('$x$', fontsize=12)
ax.set_title('DPS Sampling: Space-Time Density (log time scale)', fontsize=14)
plt.colorbar(im, ax=ax, label='Density')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'spacetime_dps.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {OUTPUT_DIR / 'spacetime_dps.png'}")

# SDE space-time density with trajectories
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(
    np.array(sde_density_log.T),
    aspect='auto',
    origin='lower',
    extent=[float(t_log_bins[0]), float(t_log_bins[-1]), float(x_bins[0]), float(x_bins[-1])],
    cmap='viridis'
)
# Overlay white trajectory lines
for i in range(n_traj_show):
    ax.plot(times_traj_log, sde_trajectory[:, i], 'w-', alpha=0.4, linewidth=0.5)
ax.set_xlabel('$\\log_{10}(t)$', fontsize=12)
ax.set_ylabel('$x$', fontsize=12)
ax.set_title('Standard SDE (True Score): Space-Time Density (log time scale)', fontsize=14)
plt.colorbar(im, ax=ax, label='Density')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'spacetime_sde.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {OUTPUT_DIR / 'spacetime_sde.png'}")

# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(
    np.array(dps_density_log.T),
    aspect='auto',
    origin='lower',
    extent=[float(t_log_bins[0]), float(t_log_bins[-1]), float(x_bins[0]), float(x_bins[-1])],
    cmap='viridis'
)
for i in range(n_traj_show):
    axes[0].plot(times_traj_log, dps_trajectory[:, i], 'w-', alpha=0.3, linewidth=0.4)
axes[0].set_xlabel('$\\log_{10}(t)$', fontsize=12)
axes[0].set_ylabel('$x$', fontsize=12)
axes[0].set_title('DPS Sampling', fontsize=14)
plt.colorbar(im0, ax=axes[0], label='Density')

im1 = axes[1].imshow(
    np.array(sde_density_log.T),
    aspect='auto',
    origin='lower',
    extent=[float(t_log_bins[0]), float(t_log_bins[-1]), float(x_bins[0]), float(x_bins[-1])],
    cmap='viridis'
)
for i in range(n_traj_show):
    axes[1].plot(times_traj_log, sde_trajectory[:, i], 'w-', alpha=0.3, linewidth=0.4)
axes[1].set_xlabel('$\\log_{10}(t)$', fontsize=12)
axes[1].set_ylabel('$x$', fontsize=12)
axes[1].set_title('Standard SDE (True Score)', fontsize=14)
plt.colorbar(im1, ax=axes[1], label='Density')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'spacetime_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {OUTPUT_DIR / 'spacetime_comparison.png'}")

Saved: figures/spacetime_dps.png


Saved: figures/spacetime_sde.png


Saved: figures/spacetime_comparison.png


In [19]:
# Compute the TRUE posterior marginal at each time for comparison (log-scaled time)
print("Computing true posterior marginal evolution...")

n_t_plot = 100
t_plot = jnp.logspace(-3, 0, n_t_plot)  # log-spaced from 0.001 to 1
x_plot = jnp.linspace(-5, 5, 200)

true_marginal_density = jnp.zeros((n_t_plot, len(x_plot)))
for i, t in enumerate(t_plot):
    marginal = get_marginal_posterior_distribution(
        float(t), post_means, post_stds, post_weights, schedule
    )
    true_marginal_density = true_marginal_density.at[i].set(marginal.pdf(x_plot))

fig, ax = plt.subplots(figsize=(10, 6))
t_plot_log = jnp.log10(t_plot)

im = ax.pcolormesh(
    np.array(t_plot_log), np.array(x_plot), np.array(true_marginal_density.T),
    cmap='viridis', shading='auto'
)
ax.set_xlabel('$\\log_{10}(t)$', fontsize=12)
ax.set_ylabel('$x$', fontsize=12)
ax.set_title('True Posterior Marginal $p_t(x|y)$ Evolution (log time scale)', fontsize=14)
plt.colorbar(im, ax=ax, label='Density')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'true_posterior_evolution.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {OUTPUT_DIR / 'true_posterior_evolution.png'}")

Computing true posterior marginal evolution...


Saved: figures/true_posterior_evolution.png


## 10. Sample Trajectory Visualization

In [20]:
# Trajectories are now shown as white lines on the space-time density plots above
# This cell is kept for additional standalone trajectory visualization if needed

print("Sample trajectories are overlaid on space-time density plots above.")

Sample trajectories are overlaid on space-time density plots above.


## Summary

### Key Takeaways

1. **DPS combines prior score with likelihood guidance** to sample from posteriors
2. **Posterior score decomposition**: $\nabla_x \log \pi(x|y) = \nabla_x \log \pi(x) + \nabla_x \log p(y|x)$
3. **Tweedie's formula** estimates $x_0$ from noisy $x_t$, enabling likelihood computation
4. For GMM prior + Gaussian likelihood, the posterior is **analytically tractable** (also a GMM)
5. DPS samples **match the true posterior** when guidance is properly scaled

### Practical Notes

- In real applications, the prior score comes from a **trained diffusion model**
- Here we used the **analytical score** of the GMM marginal at each time
- The guidance scale $\lambda$ may need tuning for different problems
- Larger $T$ makes the initial distribution closer to Gaussian (easier initialization)

In [21]:
# Final summary
print("\n" + "=" * 60)
print("DPS RESULTS SUMMARY")
print("=" * 60)
print(f"  Prior: 3-component GMM at {list(prior_config['means'])}")
print(f"  Observation: y_obs = {likelihood_config['y_obs']}, sigma_y = {likelihood_config['sigma_y']}")
print(f"  Diffusion T = {diffusion_config['T']}, steps = {diffusion_config['n_steps']}")
print(f"  Samples generated: {sampling_config['n_samples']}")
print(f"  Guidance scale: {sampling_config['guidance_scale']}")
print("=" * 60)
print("\nSAVED FIGURES:")
for f in sorted(OUTPUT_DIR.glob('*.png')):
    print(f"  {f.name}")
print("=" * 60)


DPS RESULTS SUMMARY
  Prior: 3-component GMM at [Array(-2.5, dtype=float64), Array(0., dtype=float64), Array(1.5, dtype=float64)]
  Observation: y_obs = 0.5, sigma_y = 0.8
  Diffusion T = 1.0, steps = 1000
  Samples generated: 5000
  Guidance scale: 1.0

SAVED FIGURES:
  histogram_dps.png
  histogram_sde.png
  lotka_volterra_example.png
  marginal_evolution.png
  noise_schedule.png
  posterior_score_spacetime.png
  prior_distribution.png
  prior_score_spacetime.png
  prior_vs_posterior.png
  spacetime_comparison.png
  spacetime_dps.png
  spacetime_sde.png
  true_posterior_evolution.png
